# 13 — Examples Repository Training Data

Clones [github.com/arolang/Examples](https://github.com/arolang/Examples) and
turns each example into high-quality training pairs.

For every example that has a README with a `## Prompt` section:

1. **Ground-truth pair** — the original prompt → all source files.
2. **9 alternative prompts** — LLM-generated rephrasings → same source files.
3. **Snippet pairs** — each code block gets a natural-language description,
   producing small (comment, code) training pairs that teach the model to
   understand individual ARO constructs.

**Input:**  `https://github.com/arolang/Examples` (cloned to `../data/examples_repo/`)
**Output:** Appended to `../data/02_knowledge/knowledge_pairs.jsonl`

## Setup

In [ ]:
import sys, importlib, json, re, textwrap, subprocess
from pathlib import Path
from collections import Counter

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

REPO_URL  = 'https://github.com/arolang/Examples.git'
REPO_DIR  = DATA_ROOT / 'examples_repo'

# Source file extensions to include in the output
CODE_EXTENSIONS = {'.aro', '.yaml', '.yml', '.screen', '.json', '.toml'}
SKIP_FILES      = {'README.md', 'test.hint', 'expected.txt', '.gitkeep'}
SKIP_DIRS       = {'.git', '__pycache__', 'node_modules', 'output'}

print(f'Config loaded | model={MODEL_ID}')
print(f'Repo dir:  {REPO_DIR}')
print(f'Pairs:     {PAIRS_FILE}')

## Clone / update the examples repository

In [ ]:
if (REPO_DIR / '.git').exists():
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                   capture_output=True, text=True, timeout=60)
else:
    print(f'Cloning {REPO_URL} → {REPO_DIR}...')
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
                   check=True, timeout=120)

# Discover example directories from cloned repo
example_dirs = sorted(
    [d for d in REPO_DIR.iterdir()
     if d.is_dir() and not d.name.startswith('.') and (d / 'README.md').exists()],
    key=lambda d: d.name,
)
_repo_names = {d.name for d in example_dirs}
print(f'Found {len(example_dirs)} examples from cloned repo')

# Also discover local examples (EXAMPLES_DIR) that have plan.md but aren't in the repo
_local_extra = sorted(
    [d for d in EXAMPLES_DIR.iterdir()
     if d.is_dir() and not d.name.startswith('.')
     and (d / 'plan.md').exists()
     and d.name not in _repo_names],
    key=lambda d: d.name,
)
example_dirs.extend(_local_extra)
print(f'Found {len(_local_extra)} additional local examples with plan.md')
print(f'Total: {len(example_dirs)} examples')

## Load model

In [ ]:
model, tokenizer, _load_fn, mlx_generate, make_sampler = load_model()
print('Model ready.')

## Helpers

In [ ]:
SYSTEM_PROMPT = """You are an expert ARO language programmer.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }

KEY RULES:
- Articles (a/an/the) are optional everywhere
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Exactly ONE Application-Start per application
- openapi.yaml required for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.
- Return statuses: <OK: status>, <Created: status>, <NotFound: status>
"""


def chat(messages, max_tokens=800, temperature=0.5):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=temperature),
    )


def extract_prompt(readme_text):
    """Extract the prompt from a README's ## Prompt section."""
    m = re.search(r'##\s*Prompt\s*\n+>?\s*(.+?)(?=\n##|\n\n##|\Z)', readme_text, re.DOTALL)
    if not m:
        return None
    prompt = m.group(1).strip()
    # Clean up blockquote markers
    prompt = re.sub(r'^>\s*', '', prompt, flags=re.MULTILINE).strip()
    return prompt if len(prompt) >= 15 else None


def collect_code_files(app_dir):
    """Collect all source files from an example directory.
    Returns dict of {relative_path: content}."""
    files = {}
    for path in sorted(app_dir.rglob('*')):
        if not path.is_file():
            continue
        if path.name in SKIP_FILES:
            continue
        if any(part in SKIP_DIRS for part in path.parts):
            continue
        if path.suffix.lower() in CODE_EXTENSIONS:
            rel = str(path.relative_to(app_dir))
            try:
                files[rel] = path.read_text(errors='replace').rstrip()
            except Exception:
                pass
    return files


def format_code_output(code_files):
    """Format code files into the standard multi-file output format."""
    parts = []
    # Sort: openapi first, .aro next, then templates, then rest
    def sort_key(item):
        name = item[0]
        if 'openapi' in name:
            return (0, name)
        if name.endswith('.aro'):
            return (1, name)
        if name.endswith('.screen'):
            return (2, name)
        return (3, name)

    for fname, content in sorted(code_files.items(), key=sort_key):
        ext = Path(fname).suffix.lstrip('.')
        lang = {'aro': 'aro', 'yaml': 'yaml', 'yml': 'yaml',
                'screen': 'text', 'json': 'json', 'toml': 'toml'}.get(ext, ext)
        parts.append(f'## {fname}\n```{lang}\n{content}\n```')
    return '\n\n'.join(parts)


def generate_alternative_prompts(original_prompt, n=0)  # audit fix: variants without semantic-alignment hurt the model:
    """Generate n alternative phrasings of the original prompt."""
    messages = [
        {'role': 'system', 'content': (
            'You are a technical writer. Your task is to produce shorter, vaguer '
            'paraphrases of a requirement that keep the high-level intent but strip '
            'all implementation details.'
        )},
        {'role': 'user', 'content': textwrap.dedent(f"""
            Below is the original requirement used to build an ARO application.
            Produce exactly {n} alternative phrasings.

            Rules:
            - Each alternative describes only WHAT the app does, not HOW.
            - Strip all ARO-specific details (actions, feature sets, repositories).
            - Each should be shorter and less precise than the original.
            - Write as a developer request (\"Build me...\", \"Create...\", \"Write...\").
            - Output ONLY the {n} alternatives, one per line, numbered 1. 2. 3. ... 9.

            ORIGINAL:
            {original_prompt.strip()}
        """).strip()},
    ]
    response = chat(messages, max_tokens=400, temperature=0.8)
    alternatives = []
    for line in response.splitlines():
        m = re.match(r'^\s*\d+[.)\s]+(.+)', line)
        if m:
            text = m.group(1).strip()
            if text:
                alternatives.append(text)
    return alternatives[:n]


def extract_aro_blocks(code_files):
    """Extract individual feature set blocks from .aro files.
    Returns list of (filename, block_code) tuples."""
    blocks = []
    for fname, content in code_files.items():
        if not fname.endswith('.aro'):
            continue
        # Split on feature set boundaries: (Name: Activity) { ... }
        # Match top-level feature sets including their body
        pattern = r'(\([^)]+\)\s*(?:when\s+[^{]+)?\{)'
        parts = re.split(pattern, content)
        current_block = ''
        depth = 0
        for part in parts:
            current_block += part
            depth += part.count('{') - part.count('}')
            if depth == 0 and current_block.strip():
                cleaned = current_block.strip()
                # Only keep blocks that look like feature sets
                if cleaned.startswith('(') and '{' in cleaned:
                    blocks.append((fname, cleaned))
                current_block = ''
    return blocks


def generate_snippet_description(fname, block_code):
    """Generate a natural-language description for a code block."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'Describe what the following ARO feature set does in 1-2 sentences. '
            f'Write as if you are writing a code comment that explains the purpose '
            f'and behavior. Be specific about what actions are performed and what '
            f'data flows through the feature set.\n\n'
            f'File: {fname}\n'
            f'```aro\n{block_code[:1500]}\n```\n\n'
            f'Reply with ONLY the description. No markdown, no labels.'
        )},
    ]
    desc = chat(messages, max_tokens=150, temperature=0.3).strip()
    # Clean up common LLM prefixes
    desc = re.sub(r'^(This feature set|This handler|This)\s+', '', desc, count=1)
    desc = desc[0].upper() + desc[1:] if desc else desc
    return desc if len(desc) >= 15 else None


print('Helpers ready.')

## Main loop — process each example

In [ ]:
clean_notebook_pairs('NB12')

all_pairs = []
stats = {}

for app_dir in example_dirs:
    app_name = app_dir.name

    # ── Extract prompt from README or plan.md ────────────────────────────
    readme_path = app_dir / 'README.md'
    original_prompt = None
    if readme_path.exists():
        readme = readme_path.read_text(errors='replace')
        original_prompt = extract_prompt(readme)

    if not original_prompt:
        # Fall back to plan.md (written by the training pipeline)
        plan_path = app_dir / 'plan.md'
        # Also check local Examples/ for a matching plan.md
        local_plan = EXAMPLES_DIR / app_name / 'plan.md'
        if plan_path.exists():
            original_prompt = plan_path.read_text().strip()
        elif local_plan.exists():
            original_prompt = local_plan.read_text().strip()
        if not original_prompt:
            print(f'  {app_name:<30}  skipped (no ## Prompt and no plan.md)')
            continue

    # ── Collect source files ──────────────────────────────────────────────
    code_files = collect_code_files(app_dir)
    if not code_files:
        print(f'  {app_name:<30}  skipped (no source files)')
        continue

    code_output = format_code_output(code_files)
    app_pairs = []
    source_key = f'examples_repo:{app_name}'

    # ── 1. Ground-truth pair (original prompt → full code) ────────────────
    app_pairs.append({
        'instruction': original_prompt,
        'output': code_output,
        'source': f'{source_key}/prompt',
        'score': 1.0,
        'metadata': {'example': app_name, 'type': 'ground_truth'},
    })

    # ── 2. Generate 4 alternative prompts ─────────────────────────────────
    print(f'  {app_name:<30}  generating alternatives...', end='', flush=True)
    alternatives = generate_alternative_prompts(original_prompt, n=0)  # audit fix: variants without semantic-alignment hurt the model
    print(f'  got {len(alternatives)}', flush=True)

    for i, alt in enumerate(alternatives, 1):
        app_pairs.append({
            'instruction': alt,
            'output': code_output,
            'source': f'{source_key}/alt_{i}',
            'score': 1.0,
            'metadata': {'example': app_name, 'type': 'alternative_prompt'},
        })

    # ── 3. Snippet pairs (block-by-block descriptions) ────────────────────
    blocks = extract_aro_blocks(code_files)
    snippet_count = 0
    for fname, block_code in blocks:
        if len(block_code.strip()) < 40:
            continue
        desc = generate_snippet_description(fname, block_code)
        if desc:
            app_pairs.append({
                'instruction': desc,
                'output': f'```aro\n{block_code}\n```',
                'source': f'{source_key}/snippet_{snippet_count}',
                'score': 1.0,
                'metadata': {'example': app_name, 'file': fname, 'type': 'snippet'},
            })
            snippet_count += 1

    # ── Save pairs ────────────────────────────────────────────────────────
    save_notebook_pairs('NB12', app_pairs)
    all_pairs.extend(app_pairs)

    n_alts = len(alternatives)
    stats[app_name] = {
        'files': len(code_files),
        'blocks': len(blocks),
        'alts': n_alts,
        'snippets': snippet_count,
        'total': 1 + n_alts + snippet_count,
    }
    print(f'  {app_name:<30}  {len(code_files)} files, {len(blocks)} blocks → '
          f'{stats[app_name]["total"]} pairs  '
          f'(1 original + {n_alts} alts + {snippet_count} snippets)')

print(f'\nTotal: {len(all_pairs)} pairs from {len(stats)} examples')

## Summary

In [ ]:
print('=' * 65)
print('Examples Repository Training Data — Summary')
print('=' * 65)

types = Counter(p.get('metadata', {}).get('type', 'unknown') for p in all_pairs)

print(f'\n{"Example":<30}  {"Files":>5}  {"Blocks":>6}  {"Alts":>4}  {"Snip":>4}  {"Total":>5}')
print('─' * 65)
for name, s in sorted(stats.items()):
    print(f'{name:<30}  {s["files"]:>5}  {s["blocks"]:>6}  {s["alts"]:>4}  {s["snippets"]:>4}  {s["total"]:>5}')
print('─' * 65)
total = sum(s['total'] for s in stats.values())
print(f'{"TOTAL":<30}  {"":>5}  {"":>6}  {"":>4}  {"":>4}  {total:>5}')

print(f'\nBy pair type:')
for t, n in sorted(types.items(), key=lambda x: -x[1]):
    print(f'  {t:<25}: {n}')

print(f'\nAll pairs appended to: {PAIRS_FILE}')

In [ ]:
import csv
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_csv_out = _run_dir / '12_examples_repo_training.csv'

with open(_csv_out, 'w', newline='') as _cf:
    w = csv.writer(_cf)
    w.writerow(['example', 'prompt_count', 'has_paraphrase'])
    for _name in sorted(stats):
        s = stats[_name]
        w.writerow([_name, s['total'], s['alts'] > 0])

print(f'CSV saved: {_csv_out}  ({len(stats)} rows)')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222',
    'axes.labelcolor': '#222222',
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.titlecolor': '#111111',
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight',
    'savefig.dpi': 150,
})
import numpy as np
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '12_examples_repo_training.png'

if stats:
    _labels   = sorted(stats.keys())
    _originals = [1 for _ in _labels]
    _alts      = [stats[a]['alts'] for a in _labels]
    _snippets  = [stats[a]['snippets'] for a in _labels]
    _totals    = [stats[a]['total'] for a in _labels]

    y = np.arange(len(_labels))
    h = 0.6
    fig, ax = plt.subplots(figsize=(10, max(4, len(_labels) * 0.45 + 1.5)))

    b1 = ax.barh(y, _originals, h, label='Original prompt', color='#3498db', edgecolor='white')
    b2 = ax.barh(y, _alts, h, left=_originals, label='Alternative prompts', color='#2ecc71', edgecolor='white')
    b3 = ax.barh(y, _snippets, h,
                 left=[o + a for o, a in zip(_originals, _alts)],
                 label='Snippet pairs', color='#e67e22', edgecolor='white')

    ax.set_yticks(y)
    ax.set_yticklabels(_labels, fontsize=8)
    ax.set_xlabel('Training pairs')
    ax.set_title(
        f'Examples Repo Training — {sum(_totals):,} pairs from {len(_labels)} examples',
        fontsize=13, fontweight='bold'
    )
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(axis='x', alpha=0.3)

    for i, tot in enumerate(_totals):
        ax.text(tot + 0.15, i, str(tot), va='center', ha='left', fontsize=8, color='#333')

    fig.tight_layout()
    fig.savefig(_out)
    plt.close(fig)
    print(f'Saved: {_out}')
else:
    print('No data to plot.')